# AttackAware PolyIoM v1.1.4 — full-knowledge inversion

The ablation showed the polynomial **costs** recognition accuracy on voice
(EER 1.93% against 0.27% without it, firm) and buys no measurable
unlinkability. The one property that could still justify it is the property
Stage A actually selected for: **resistance to inversion**. That has never
been measured on the final system. This notebook measures it.

**Adversary: level 3, full knowledge.** Holds `K*`, `R`, the algorithm *and*
the stored template `h`. This is the worst case, and the one an
irreversibility claim has to survive — an attacker who breached the database
has the template.

**The attack.** Annealed-softmax gradient descent on `z'`, constrained to the
unit sphere, cross-entropy against the stored bucket indices. The polynomial
is differentiable, so gradients flow through it exactly as through the linear
arms. **The same attack at the same budget runs against all three arms** — 5
restarts × 800 Adam steps, fixed in advance. A scheme can always be made to
look secure by attacking it weakly.

**Headline metric: SAR@τ\*** — the fraction of templates whose reconstruction
is *accepted* at the sealed threshold.

**The polynomial helps** only if SAR is firmly lower for `polyiom` than for
both baselines under the paired identity bootstrap. Success criteria were
fixed before running; see `THREAT_MODEL.md`.

**What this measures:** practical inversion resistance under a stated budget.
It is *not* information-theoretic irreversibility — a perfect solution
provably exists in the search space, since `z_true` reproduces `h` exactly, so
any failure to find it is an optimisation-hardness result and is reported in
those words.

**Three assertions run before any number is reported:** the differentiable
polynomial reproduces the study's to float tolerance; the attack targets equal
the stored templates; and a random unit vector scores near the `M/q` chance
rate, which anchors the SAR scale.


In [ ]:
#@title 1. Mount Drive and verify the inversion inputs
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, math, os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4")
DIR = {
    "protocol": PROJECT / "protocol",
    "embeddings": PROJECT / "embeddings",
    "runs": PROJECT / "runs",
    "seal": PROJECT / "seal",
}
S0, G = 2026, 5
DEVICE = torch.device("cpu")

LFW_FACE_TRIALS = DIR["protocol"] / "lfw_face_trials.tsv"
LFW_EMB = DIR["embeddings"] / "lfw_all_valid_embeddings.npz"
LIBRI_MANIFEST = DIR["protocol"] / "librispeech_internal.tsv"
LIBRI_EMB = DIR["embeddings"] / "librispeech_internal_embeddings.npz"

HELDOUT_RESULTS = {
    modality: DIR["runs"] / "heldout" / modality / "heldout_result.json"
    for modality in ("face", "voice")
}
ABLATION_OUTPUT = DIR["runs"] / "ablation" / "ablation_result.json"

REQUIRED = [
    LFW_FACE_TRIALS, LFW_EMB, LIBRI_MANIFEST, LIBRI_EMB,
    DIR["runs"] / "key_search/face/selected_key.json",
    DIR["runs"] / "key_search/voice/selected_key.json",
    HELDOUT_RESULTS["face"], HELDOUT_RESULTS["voice"],
    PROJECT / "ablation_only.py",
    PROJECT / "inversion_only.py",
]
missing = [str(p.relative_to(PROJECT)) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Inversion cannot start; required artefacts are missing:\n- "
        + "\n- ".join(missing)
    )

torch.use_deterministic_algorithms(True, warn_only=False)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print("Preflight: PASS")
print("Device:", DEVICE)
print("Writes exactly one file: runs/inversion/inversion_result.json")
print("Changes no seal and fits no threshold.")

In [ ]:
#@title 2. Load the frozen-pipeline primitives and the attack
for name in ("ablation_only.py", "inversion_only.py"):
    path = PROJECT / name
    exec(compile(path.read_text(), str(path), "exec"), globals())
print("Inversion runtime: READY")

In [ ]:
#@title 3. Run the full-knowledge inversion attack
inversion = run_inversion()
inversion_report(inversion)